# 🧠 VGGNet-19 | Brain Tumor MRI Classification
**Framework:** PyTorch  
**Dataset:** 7,200 MRI images — Glioma · Meningioma · Pituitary · No Tumor  
**Split:** 70% Train | 15% Validation | 15% Test (Stratified)  
**Metrics:** Accuracy · Precision · Recall · F1 · AUC-ROC · Confusion Matrix · Loss/Acc Curves

---
### ⚡ Before running: enable GPU
`Runtime` → `Change runtime type` → **T4 GPU** → Save

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️  No GPU! Go to Runtime → Change runtime type → T4 GPU')

---
### 📦 Step 1 — Install libraries

In [ ]:
!pip install -q torch torchvision scikit-learn seaborn matplotlib Pillow
print('✅ Libraries ready')

---
### 📂 Step 2 — Mount Google Drive
Upload your dataset to Drive first. Expected structure:
```
MyDrive/
  brain-tumor-mri/
    Training/  glioma/  meningioma/  notumor/  pituitary/
    Testing/   glioma/  meningioma/  notumor/  pituitary/
```

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

# ✏️  UPDATE THESE PATHS
TRAIN_DIR = '/content/drive/MyDrive/brain-tumor-mri/Training'
TEST_DIR  = '/content/drive/MyDrive/brain-tumor-mri/Testing'

for d in [TRAIN_DIR, TEST_DIR]:
    if os.path.isdir(d):
        print(f'✅ Found: {d}  classes: {sorted(os.listdir(d))}')
    else:
        print(f'❌ NOT FOUND: {d}  <- fix path above')

---
### ⚙️ Step 3 — Imports & Configuration

In [ ]:
import os, random, time, warnings
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import VGG19_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, auc,
    confusion_matrix, classification_report
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')

class Config:
    DATASET_DIR     = TRAIN_DIR
    EXTRA_DIR       = TEST_DIR
    MERGE_DIRS      = True
    OUTPUT_DIR      = '/content/outputs'
    CLASSES         = ['glioma', 'meningioma', 'notumor', 'pituitary']
    NUM_CLASSES     = 4
    IMG_SIZE        = 224
    TRAIN_RATIO     = 0.70
    VAL_RATIO       = 0.15
    TEST_RATIO      = 0.15
    BATCH_SIZE      = 32
    EPOCHS          = 15
    LR              = 1e-4
    FINE_TUNE_LR    = 1e-5
    FINE_TUNE_EPOCH = 10
    PATIENCE        = 5
    DROPOUT         = 0.5
    WEIGHT_DECAY    = 1e-4
    NUM_WORKERS     = 2
    DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    SEED            = 42

cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

random.seed(cfg.SEED); np.random.seed(cfg.SEED); torch.manual_seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.SEED)
    torch.backends.cudnn.deterministic = True

print(f'Device : {cfg.DEVICE}')
print(f'Epochs : {cfg.EPOCHS} + {cfg.FINE_TUNE_EPOCH} fine-tune')
print('✅ Config ready')

---
### 🔄 Step 4 — Transforms

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=10),
    transforms.RandomPerspective(distortion_scale=0.1, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
print('✅ Transforms defined')

---
### 🗂️ Step 5 — Dataset (Stratified 70/15/15 split)

In [ ]:
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]   # PIL Image (transform=None on base ds)
        if self.transform:
            img = self.transform(img)
        return img, label

def build_datasets():
    if cfg.MERGE_DIRS and cfg.EXTRA_DIR and os.path.isdir(cfg.EXTRA_DIR):
        ds1 = datasets.ImageFolder(cfg.DATASET_DIR, transform=None)
        ds2 = datasets.ImageFolder(cfg.EXTRA_DIR,   transform=None)
        assert ds1.class_to_idx == ds2.class_to_idx, 'Class mismatch!'
        full_ds    = torch.utils.data.ConcatDataset([ds1, ds2])
        all_labels = ds1.targets + ds2.targets
        print(f'  Merged: {len(full_ds)} images | map: {ds1.class_to_idx}')
    else:
        full_ds    = datasets.ImageFolder(cfg.DATASET_DIR, transform=None)
        all_labels = full_ds.targets
        print(f'  Dataset: {len(full_ds)} images | map: {full_ds.class_to_idx}')

    all_labels = np.array(all_labels)
    indices    = np.arange(len(full_ds))

    train_val_idx, test_idx = train_test_split(
        indices, test_size=cfg.TEST_RATIO,
        stratify=all_labels[indices], random_state=cfg.SEED)

    val_frac = cfg.VAL_RATIO / (cfg.TRAIN_RATIO + cfg.VAL_RATIO)
    train_idx, val_idx = train_test_split(
        train_val_idx, test_size=val_frac,
        stratify=all_labels[train_val_idx], random_state=cfg.SEED)

    print(f'  Train : {len(train_idx)} | Val : {len(val_idx)} | Test : {len(test_idx)}')
    print(f'  Train dist: {dict(Counter(all_labels[train_idx]))}')

    train_set = TransformSubset(Subset(full_ds, train_idx), train_transform)
    val_set   = TransformSubset(Subset(full_ds, val_idx),   eval_transform)
    test_set  = TransformSubset(Subset(full_ds, test_idx),  eval_transform)
    return train_set, val_set, test_set, all_labels[train_idx]

train_set, val_set, test_set, train_labels = build_datasets()

train_loader = DataLoader(train_set, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True)
print('✅ DataLoaders ready')

---
### 👁️ Step 6 — Preview sample images

In [ ]:
import torchvision
images, labels = next(iter(train_loader))
mean_t = torch.tensor(IMAGENET_MEAN).view(3,1,1)
std_t  = torch.tensor(IMAGENET_STD).view(3,1,1)
imgs_show = (images[:8] * std_t + mean_t).clamp(0, 1)
grid = torchvision.utils.make_grid(imgs_show, nrow=4, padding=4)
fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(grid.permute(1, 2, 0))
ax.set_title('Sample Training Images (augmented)', fontsize=13)
ax.axis('off')
plt.savefig(f'{cfg.OUTPUT_DIR}/sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print('Labels:', [cfg.CLASSES[l] for l in labels[:8].tolist()])

---
### 🏗️ Step 7 — VGGNet-19 Model

In [ ]:
class VGG19BrainTumor(nn.Module):
    def __init__(self, num_classes=4, dropout=0.5):
        super().__init__()
        backbone      = models.vgg19(weights=VGG19_Weights.IMAGENET1K_V1)
        self.features = backbone.features
        self.avgpool  = backbone.avgpool
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512*7*7, 4096),
            nn.BatchNorm1d(4096), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(4096, 4096),
            nn.BatchNorm1d(4096), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(4096, 1024),
            nn.BatchNorm1d(1024), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(1024, num_classes),
        )
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.classifier(self.avgpool(self.features(x)))

    def freeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad = False

    def unfreeze_blocks(self, from_block=4):
        starts = {1:0, 2:5, 3:10, 4:19, 5:28}
        for i, layer in enumerate(self.features):
            if i >= starts.get(from_block, 19):
                for p in layer.parameters(): p.requires_grad = True
        n = sum(p.numel() for p in self.features.parameters() if p.requires_grad)
        print(f'  Unfrozen backbone params: {n:,}')

model = VGG19BrainTumor(cfg.NUM_CLASSES, cfg.DROPOUT).to(cfg.DEVICE)
print(f'  Total params     : {sum(p.numel() for p in model.parameters()):,}')
print(f'  Trainable params : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print('✅ Model ready')

---
### ⚖️ Step 8 — Class weights & loss

In [ ]:
classes_arr = np.unique(train_labels)
weights     = compute_class_weight('balanced', classes=classes_arr, y=train_labels)
class_weights = torch.tensor(weights, dtype=torch.float32).to(cfg.DEVICE)
print(f'  Weights: {dict(zip(cfg.CLASSES, weights.round(3)))}')
criterion = nn.CrossEntropyLoss(weight=class_weights)
print('✅ Loss ready')

---
### 🚂 Step 9 — Training loop

In [ ]:
def run_epoch(model, loader, criterion, optimizer, is_train):
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(cfg.DEVICE), labels.to(cfg.DEVICE)
            if is_train: optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += images.size(0)
    return total_loss / total, correct / total

# -- Phase 1: frozen backbone --
model.freeze_backbone()
optimizer1 = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler1 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer1, mode='min', factor=0.5, patience=3)

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val_acc, patience_count = 0.0, 0
best_path = f'{cfg.OUTPUT_DIR}/best_model.pth'

print('-'*60)
print('  PHASE 1 -- Head training (backbone frozen)')
print('-'*60)

for epoch in range(1, cfg.EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer1, True)
    vl_loss, vl_acc = run_epoch(model, val_loader,   criterion, optimizer1, False)
    scheduler1.step(vl_loss)
    for k, v in zip(['train_loss','val_loss','train_acc','val_acc'],
                    [tr_loss, vl_loss, tr_acc, vl_acc]):
        history[k].append(v)
    flag = ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), best_path)
        patience_count = 0; flag = '  <- best'
    else:
        patience_count += 1
    print(f'  Ep {epoch:>2}/{cfg.EPOCHS} | Loss {tr_loss:.4f}/{vl_loss:.4f} | '
          f'Acc {tr_acc*100:.1f}%/{vl_acc*100:.1f}% | {time.time()-t0:.1f}s{flag}')
    if patience_count >= cfg.PATIENCE:
        print(f'  Early stop at epoch {epoch}.'); break

print(f'\n  Phase 1 best val acc: {best_val_acc*100:.2f}%')

---
### 🔧 Step 10 — Fine-tune Phase 2

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=cfg.DEVICE))
model.unfreeze_blocks(from_block=4)

optimizer2 = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=cfg.FINE_TUNE_LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler2 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer2, T_max=cfg.FINE_TUNE_EPOCH, eta_min=1e-7)

patience_count = 0
print('-'*60)
print('  PHASE 2 -- Fine-tuning (blocks 4 & 5 unfrozen)')
print('-'*60)

for epoch in range(1, cfg.FINE_TUNE_EPOCH + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer2, True)
    vl_loss, vl_acc = run_epoch(model, val_loader,   criterion, optimizer2, False)
    scheduler2.step()
    for k, v in zip(['train_loss','val_loss','train_acc','val_acc'],
                    [tr_loss, vl_loss, tr_acc, vl_acc]):
        history[k].append(v)
    flag = ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), best_path)
        patience_count = 0; flag = '  <- best'
    else:
        patience_count += 1
    print(f'  FT {epoch:>2}/{cfg.FINE_TUNE_EPOCH} | Loss {tr_loss:.4f}/{vl_loss:.4f} | '
          f'Acc {tr_acc*100:.1f}%/{vl_acc*100:.1f}% | {time.time()-t0:.1f}s{flag}')
    if patience_count >= cfg.PATIENCE:
        print(f'  Early stop at FT epoch {epoch}.'); break

model.load_state_dict(torch.load(best_path, map_location=cfg.DEVICE))
torch.save(model.state_dict(), f'{cfg.OUTPUT_DIR}/vgg19_final.pth')
print(f'\n  Training complete. Best val acc: {best_val_acc*100:.2f}%')
print('✅ Model saved')

---
### 📈 Step 11 — Loss & Accuracy Curves

In [ ]:
epochs_x = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('VGGNet-19 Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(epochs_x, history['train_loss'], 'b-o', ms=4, lw=2, label='Train')
axes[0].plot(epochs_x, history['val_loss'],   'r-o', ms=4, lw=2, label='Validation')
axes[0].axvline(x=cfg.EPOCHS, color='gray', ls='--', lw=1.5, label='Fine-tune start')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, [a*100 for a in history['train_acc']], 'b-o', ms=4, lw=2, label='Train')
axes[1].plot(epochs_x, [a*100 for a in history['val_acc']],   'r-o', ms=4, lw=2, label='Validation')
axes[1].axvline(x=cfg.EPOCHS, color='gray', ls='--', lw=1.5, label='Fine-tune start')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)'); axes[1].set_ylim(0, 105)
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{cfg.OUTPUT_DIR}/loss_accuracy_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: loss_accuracy_curves.png')

---
### 🧪 Step 12 — Evaluate on Test Set

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    for images, labels in loader:
        images  = images.to(cfg.DEVICE)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1).cpu().numpy()
        preds   = outputs.argmax(1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

y_true, y_pred, y_prob = evaluate(model, test_loader)
y_bin = label_binarize(y_true, classes=list(range(cfg.NUM_CLASSES)))

acc   = accuracy_score(y_true, y_pred)
prec  = precision_score(y_true, y_pred, average='weighted', zero_division=0)
rec   = recall_score(y_true, y_pred,    average='weighted', zero_division=0)
f1    = f1_score(y_true, y_pred,        average='weighted', zero_division=0)
auc_v = roc_auc_score(y_bin, y_prob,    multi_class='ovr', average='weighted')

print('='*50)
print('  TEST SET RESULTS')
print('='*50)
print(f'  Accuracy  : {acc*100:.2f}%')
print(f'  Precision : {prec:.4f}  (weighted)')
print(f'  Recall    : {rec:.4f}  (weighted)')
print(f'  F1-Score  : {f1:.4f}  (weighted)')
print(f'  AUC-ROC   : {auc_v:.4f}  (weighted OvR)')
print('='*50)
print()
print(classification_report(y_true, y_pred, target_names=cfg.CLASSES, digits=4))

---
### 🗺️ Step 13 — Confusion Matrix

In [ ]:
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Confusion Matrix -- VGGNet-19', fontsize=13, fontweight='bold')

for ax, data, fmt, title in zip(
    axes, [cm, cm_norm], ['d', '.2%'], ['Raw Counts', 'Normalized']
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=cfg.CLASSES, yticklabels=cfg.CLASSES,
                ax=ax, linewidths=0.5, linecolor='white', annot_kws={'size':11})
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(f'{cfg.OUTPUT_DIR}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: confusion_matrix.png')

---
### 📉 Step 14 — AUC-ROC Curves

In [ ]:
PALETTE = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
fig, ax = plt.subplots(figsize=(9, 7))

for i, (cls, color) in enumerate(zip(cfg.CLASSES, PALETTE)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{cls.capitalize()} (AUC={roc_auc:.4f})')

fpr_m, tpr_m, _ = roc_curve(y_bin.ravel(), y_prob.ravel())
ax.plot(fpr_m, tpr_m, 'k--', lw=2, label=f'Micro-avg (AUC={auc(fpr_m,tpr_m):.4f})')
ax.plot([0,1],[0,1],'gray',lw=1,ls=':')
ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate',  fontsize=12)
ax.set_title('AUC-ROC Curves (One-vs-Rest)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{cfg.OUTPUT_DIR}/auc_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: auc_roc_curves.png')

---
### 📊 Step 15 — Per-Class Metrics Bar Chart

In [ ]:
prec_pc = precision_score(y_true, y_pred, average=None, zero_division=0)
rec_pc  = recall_score(y_true, y_pred,    average=None, zero_division=0)
f1_pc   = f1_score(y_true, y_pred,        average=None, zero_division=0)
acc_pc  = cm.diagonal() / cm.sum(axis=1)

x, w = np.arange(cfg.NUM_CLASSES), 0.2
fig, ax = plt.subplots(figsize=(12, 6))

for offset, vals, label, color in [
    (-1.5*w, acc_pc,  'Accuracy',  '#1f77b4'),
    (-0.5*w, prec_pc, 'Precision', '#ff7f0e'),
    ( 0.5*w, rec_pc,  'Recall',    '#2ca02c'),
    ( 1.5*w, f1_pc,   'F1-Score',  '#d62728'),
]:
    bars = ax.bar(x + offset, vals*100, w, label=label, color=color)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.5,
                f'{h:.1f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels([c.capitalize() for c in cfg.CLASSES], fontsize=11)
ax.set_ylim(0, 115); ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Per-Class Metrics -- VGGNet-19', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{cfg.OUTPUT_DIR}/per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: per_class_metrics.png')

---
### 💾 Step 16 — Download all outputs

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/vgg19_outputs', 'zip', cfg.OUTPUT_DIR)
files.download('/content/vgg19_outputs.zip')
print('✅ Download started')